In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
# !pip install pyqt5 --quiet # For Qt5Agg
# !pip install autoreject --quiet

import mne
import os, gc, numpy as np, pandas as pd, scipy.signal as sig
# from scipy.signal import hilbert
# from autoreject import AutoReject
import matplotlib.pyplot as plt
import matplotlib
from google.colab import files
# import plotly.graph_objects as go

# Set path
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData/'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()])
# print(subject_dirs)

output_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/ERPs_Analysis/ICA_Component_Images'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37','E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31','E106', 'E112', 'E80','E54', 'E55', 'E79'],
    'Right Central':     ['E87','E93', 'E98', 'E102', 'E103','E107','E108', 'E104', 'E105','E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60','E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92','E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101'] }

In [ ]:
# Which event marks the start and end of sessions?

for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # print(raw.annotations.description)

      # Create a copy of the raw data for plotting with filtered events
      raw_plot_copy = raw.copy()

      # Set standard montage on the copy
      raw_plot_copy.set_montage('GSN-HydroCel-128')

      raw_plot_copy.info["bads"] = []

      # Filter events to only include the desired types
      events, event_id = mne.events_from_annotations(raw_plot_copy) # Extract from the copy
      desired_event_names = ['Wd2E']
      # Map desired event names to their IDs
      desired_event_ids = [event_id.get(name) for name in desired_event_names if name in event_id]

      # Get the indices of the desired events
      event_indices = np.where(np.isin(events[:, 2], desired_event_ids))[0]

      # Count and print the number of 'SESS' events
      num_sess_events = len(event_indices)
      print(f"Number of 'Wd2E' events found: {num_sess_events}")


      # Create a new set of annotations with only the desired events
      filtered_annotations = mne.Annotations(
          onset=events[event_indices, 0] / raw_plot_copy.info['sfreq'],
          duration=np.zeros(len(event_indices)),
          # Access descriptions using the indices from the original annotations object
          description=[raw_plot_copy.annotations.description[idx] for idx in event_indices]
      )

      # Replace the copy's annotations with the filtered annotations
      raw_plot_copy.set_annotations(filtered_annotations)

      # Plot with the raw_plot_copy object that only has filtered annotations
      for region_name, region_channels in regions.items():
        # Use channels from the copy
        picks = [ch for ch in region_channels if ch in raw_plot_copy.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20 to the copy
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          # Check against the copy's channel names
          if dummy_name not in raw_plot_copy.ch_names:
            dummy_data = np.zeros((1, raw_plot_copy.n_times))
            dummy_info = mne.create_info([dummy_name], raw_plot_copy.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw_plot_copy.add_channels([dummy_raw], force_update_info=True) # Add to the copy
          picks.append(dummy_name)
          dummy_index += 1


        # Plot with the modified raw_plot_copy object
        fig = raw_plot_copy.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, duration=90*60)

In [ ]:
# Separate sessions from breaks

for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # # Get sampling rate
      # sfreq = raw.info['sfreq']

      # Get annotations
      bgin_annot = [ann for ann in raw.annotations if ann['description'] == 'bgin']
      bgin_onsets = np.array([ann['onset'] for ann in bgin_annot])  # in seconds

      # Compute time differences between consecutive 'bgin'
      time_diffs = np.diff(bgin_onsets)

      # Threshold for session boundary
      gap_threshold = 250  # seconds
      gap_indices = np.where(time_diffs > gap_threshold)[0]

      # Calculate start/end indices for each session
      session_starts = np.insert(bgin_onsets[gap_indices + 1], 0, bgin_onsets[0])
      session_ends = np.append(bgin_onsets[gap_indices], bgin_onsets[-1])

      # Add margin to crop (avoid edge artifacts)
      margin = 2  # seconds
      session_starts = np.maximum(session_starts - margin, 0)
      session_ends = np.minimum(session_ends + margin, raw.times[-1])

      # Sanity check
      print("Detected session boundaries:")
      for i, (start, end) in enumerate(zip(session_starts, session_ends)):
          print(f"Session {i+1}: {start:.1f}s – {end:.1f}s")

      # Crop and save each session
      for i, (start, end) in enumerate(zip(session_starts, session_ends)):
          session_raw = raw.copy().crop(tmin=start, tmax=end)
          base_name = os.path.splitext(session_file)[0]
          out_file = os.path.join(subj_path, f'{base_name}_session_auto_{i+1}.fif')
          session_raw.save(out_file, overwrite=True)
          print(f"Saved {out_file}")

In [ ]:
# Plot cleaned sessions - auto scaling

for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_file}")

      # Set GSN montage
      raw.set_montage('GSN-HydroCel-128', match_case=False)

      for region_name, region_channels in regions.items():
          picks = [ch for ch in region_channels if ch in raw.ch_names]

          if not picks:
              print(f"No valid channels found for {region_name}.")
              continue

          # Add dummy channels to get exactly 20 for plotting
          dummy_index = 1
          while len(picks) < 20:
              dummy_name = f'dummy{dummy_index}'
              if dummy_name not in raw.ch_names:
                  dummy_data = np.zeros((1, raw.n_times))
                  dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
                  dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                  raw.add_channels([dummy_raw], force_update_info=True)
              picks.append(dummy_name)
              dummy_index += 1

          # Plot EEG segment for the region
          print(f"{os.path.basename(session_path)} - {region_name}")
          fig = raw.plot(
              picks=picks,
              n_channels=20,
              scalings='auto',
              title=f"{os.path.basename(session_path)} - {region_name}",
              group_by='original',
              show_scrollbars=False,
              show_scalebars=False,
              duration=60*60)

In [ ]:
# Plot cleaned sessions - 10e-4 scaling

for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_file}")

      # Set GSN montage
      raw.set_montage('GSN-HydroCel-128', match_case=False)

      # Compute correlation matrix
      correlation_threshold = 0.2
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}")

      # save
      save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      plt.savefig(save_path)
      plt.close()

      for region_name, region_channels in regions.items():
          picks = [ch for ch in region_channels if ch in raw.ch_names]

          if not picks:
              print(f"No valid channels found for {region_name}.")
              continue

          # Add dummy channels to get exactly 20 for plotting
          dummy_index = 1
          while len(picks) < 20:
              dummy_name = f'dummy{dummy_index}'
              if dummy_name not in raw.ch_names:
                  dummy_data = np.zeros((1, raw.n_times))
                  dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
                  dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                  raw.add_channels([dummy_raw], force_update_info=True)
              picks.append(dummy_name)
              dummy_index += 1

          # Plot EEG segment for the region
          print(f"{os.path.basename(session_path)} - {region_name}")
          fig = raw.plot(
              picks=picks,
              n_channels=20,
              scalings=10e-4,
              title=f"{os.path.basename(session_path)} - {region_name}",
              group_by='original',
              show_scrollbars=False,
              show_scalebars=False,
              duration=60*60)

In [ ]:
# Separate sessions from breaks - 24

for subj in subject_dirs:
  # 24
  if int(subj) == 24:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Get sampling rate
      sfreq = raw.info['sfreq']

      # Get annotations and focus on 'bgin'
      bgin_annot = [ann for ann in raw.annotations if ann['description'] == 'bgin']
      bgin_onsets = np.array([ann['onset'] for ann in bgin_annot])  # in seconds

      # Compute time differences between consecutive 'bgin'
      time_diffs = np.diff(bgin_onsets)

      # Threshold for session boundary
      gap_threshold = 250  # seconds
      gap_indices = np.where(time_diffs > gap_threshold)[0]

      # Calculate start/end indices for each session
      session_starts = np.insert(bgin_onsets[gap_indices + 1], 0, bgin_onsets[0])
      session_ends = np.append(bgin_onsets[gap_indices], bgin_onsets[-1])

      # Add margin to crop (avoid edge artifacts)
      margin = 2  # seconds
      session_starts = np.maximum(session_starts - margin, 0)
      session_ends = np.minimum(session_ends + margin, raw.times[-1])

      # Sanity check
      print("Detected session boundaries:")
      for i, (start, end) in enumerate(zip(session_starts, session_ends)):
          print(f"Session {i+1}: {start:.1f}s – {end:.1f}s")

      # Crop and save each session
      for i, (start, end) in enumerate(zip(session_starts, session_ends)):
          session_raw = raw.copy().crop(tmin=start, tmax=end)
          base_name = os.path.splitext(session_file)[0]
          out_file = os.path.join(subj_path, f'{base_name}_session_auto_{i+1}.fif')
          session_raw.save(out_file, overwrite=True)
          print(f"Saved {out_file}")

In [ ]:
# Plot cleaned sessions - 10e-4 scaling - subject 24

for subj in subject_dirs:
  # 24
  if int(subj) == 24:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_file}")

      # Set GSN montage
      raw.set_montage('GSN-HydroCel-128', match_case=False)

      # Compute correlation matrix
      correlation_threshold = 0.2
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - {session_file}")

      # save
      save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      plt.savefig(save_path)
      plt.close()

      for region_name, region_channels in regions.items():
          picks = [ch for ch in region_channels if ch in raw.ch_names]

          if not picks:
              print(f"No valid channels found for {region_name}.")
              continue

          # Add dummy channels to get exactly 20 for plotting
          dummy_index = 1
          while len(picks) < 20:
              dummy_name = f'dummy{dummy_index}'
              if dummy_name not in raw.ch_names:
                  dummy_data = np.zeros((1, raw.n_times))
                  dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
                  dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                  raw.add_channels([dummy_raw], force_update_info=True)
              picks.append(dummy_name)
              dummy_index += 1

          # Plot EEG segment for the region
          print(f"{os.path.basename(session_path)} - {region_name}")
          fig = raw.plot(
              picks=picks,
              n_channels=20,
              scalings=10e-4,
              title=f"{os.path.basename(session_path)} - {region_name}",
              group_by='original',
              show_scrollbars=False,
              show_scalebars=False,
              duration=120*60)

In [ ]:
# 24: re-ref, filter, ICA -> topomap + time domain
for subj in subject_dirs:
  # 24
  if int(subj) == 24:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_file}")

      # Set GSN montage
      raw.set_montage('GSN-HydroCel-128', match_case=False)

      for region_name, region_channels in regions.items():
          picks = [ch for ch in region_channels if ch in raw.ch_names]

          if not picks:
              print(f"No valid channels found for {region_name}.")
              continue

      # Re-referencing
      try:
        raw.set_eeg_reference(ref_channels=['E57', 'E100'])
        print(f"{subj}: Re-referenced using LM/RM average.")
      except Exception as e:
        print(f"{subj}: Error during referencing - {e}")

      # ICA on re-referenced, filtered EEG
      try:
        raw.filter(0.1, None) # High pass filtered
      except Exception as e:
        print(f"{subj}: Error during filtering - {e}")

      # Fit ICA
      try:
        ica_filtered = mne.preprocessing.ICA(random_state=97, method='fastica')
        ica_filtered.fit(raw)
      except Exception as e:
        print(f"{subj}: Error during ICA fitting - {e}")

      # ICA plots
      try:
        print(f"{subj}: ICA topomap:")
        fig_topo = ica_filtered.plot_components(inst=raw, picks=range(20), title=f"{session_file} - non-interpolated ICA topomap", colorbar=True, ch_type='eeg')
        fig_path_topo = os.path.join(output_dir, f"{session_file}_ica_topo.png")
        fig_topo.savefig(fig_path_topo)
        plt.close(fig_topo)
        print(f"{subj}: Saved ICA topomap to {fig_path_topo}")

        print(f"{subj}: ICA component time series plot:")
        fig_time = ica_filtered.plot_sources(inst=raw, picks=range(20), start=5, end=35, show_scrollbars=False, title=f"{session_file} - non-interpolated ICA time domain")
        fig_path_time = os.path.join(output_dir, f"{session_file}_ica_time_20s.png")
        fig_time.savefig(fig_path_time)
        plt.close(fig_time)
        print(f"{subj}: Saved ICA time series to {fig_path_time}")

      except Exception as e:
        print(f"{subj}: Error during ICA plotting or saving - {e}")

In [ ]:
# 24: flag+interpolate bad channels, re-ref, filter, ICA -> topomap + time domain
for subj in subject_dirs:
  # 24
  if int(subj) == 24:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('auto_2.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_file}")

      # Set GSN montage
      raw.set_montage('GSN-HydroCel-128', match_case=False)

      # Mark bad channels
      raw2 = raw.copy()
      raw2.info["bads"] = []
      raw2.info["bads"].append('E88')

      for region_name, region_channels in regions.items():
          picks = [ch for ch in region_channels if ch in raw.ch_names]

          if not picks:
              print(f"No valid channels found for {region_name}.")
              continue

      # # Add dummy channels to get exactly 20 for plotting to raw2
      # dummy_index = 1
      # while len(picks) < 20:
      #   dummy_name = f'dummy{dummy_index}'
      #   if dummy_name not in raw2.ch_names:
      #     dummy_data = np.zeros((1, raw2.n_times))
      #     dummy_info = mne.create_info([dummy_name], raw2.info['sfreq'], ch_types='eeg')
      #     dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
      #     raw2.add_channels([dummy_raw], force_update_info=True)
      #   picks.append(dummy_name)
      #   dummy_index += 1

      # # Plot
      # fig = raw2.plot(picks=picks, n_channels=20, scalings=10e-4,
      #                 title=f"{session_file} - {region_name}", group_by='original',
      #                 show_scrollbars=False,
      #                 show_scalebars=False, duration=180*60)

      data_interp = raw2.copy().interpolate_bads(reset_bads=False)

      # for title, data in zip(["orig.", "interp."], [raw2, data_interp]):
      #   with mne.viz.use_browser_backend("matplotlib"):
      #     fig = data.plot(butterfly=True, color="#00000022", bad_color="r")
      #   fig.subplots_adjust(top=0.9)
      #   fig.suptitle(title, size="xx-large", weight="bold")

      # Re-referencing
      try:
        data_interp.set_eeg_reference(ref_channels=['E57', 'E100'])
        print(f"{subj}: Re-referenced using LM/RM average.")
      except Exception as e:
        print(f"{subj}: Error during referencing - {e}")

      # ICA on re-referenced, filtered EEG
      try:
        data_interp.filter(0.1, None) # High pass filtered
      except Exception as e:
        print(f"{subj}: Error during filtering - {e}")

      # Fit ICA
      try:
        ica_filtered = mne.preprocessing.ICA(random_state=97, method='fastica')
        ica_filtered.fit(data_interp)
      except Exception as e:
        print(f"{subj}: Error during ICA fitting - {e}")

      # ICA plots
      try:
        print(f"{subj}: ICA topomap:")
        fig_topo = ica_filtered.plot_components(inst=data_interp, picks=range(20), title=f"{session_file} - interpolated ICA topomap", colorbar=True, ch_type='eeg')
        fig_path_topo = os.path.join(output_dir, f"{session_file}_interp_ica_topo.png")
        fig_topo.savefig(fig_path_topo)
        plt.close(fig_topo)
        print(f"{subj}: Saved ICA topomap to {fig_path_topo}")

        print(f"{subj}: ICA component time series plot:")
        fig_time = ica_filtered.plot_sources(inst=data_interp, picks=range(20), title=f"{session_file} - interpolated ICA time domain")
        fig_path_time = os.path.join(output_dir, f"{session_file}_interp_ica_time.png")
        fig_time.savefig(fig_path_time)
        plt.close(fig_time)
        print(f"{subj}: Saved ICA time series to {fig_path_time}")

      except Exception as e:
        print(f"{subj}: Error during ICA plotting or saving - {e}")

In [ ]:
# raw plot - 20
for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # # Compute correlation matrix
      # correlation_threshold = 0.6
      # # Extract data as a NumPy array before computing the correlation
      # data = raw.get_data(picks='eeg')
      # ch_names = raw.info['ch_names']
      # corr_matrix = np.corrcoef(data)

      # # Calculate mean correlation for each channel (excluding self-correlation)
      # mean_corrs = []
      # for i in range(corr_matrix.shape[0]):
      #   mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
      #   mean_corrs.append(mean_corr)

      # # Identify bad channels
      # bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      # bad_channels = [ch_names[i] for i in bad_channel_indices]

      # # Output results
      # print(f"Correlation threshold: {correlation_threshold}")
      # print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      # print(bad_channels)

      # # Plot correlation matrix
      # plt.figure(figsize=(16, 14))  # Increased figure size
      # plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      # plt.colorbar(label='Correlation Coefficient')
      # plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, duration=180*60)

      # add bad channels to list -> save the plot with different color for bad channels

      # # Save figure
      # save_path = os.path.join(output_dir, f"{session_file}_{region_name}.png")
      # fig.savefig(save_path)
      # plt.close(fig)

      # # Save bad channels
      # bads_path = os.path.join(output_dir, f"{session_file}_bads.txt")
      # with open(bads_path, 'w') as f:
      # for bad in raw.info['bads']:
      # f.write(f"{bad}\n")

In [ ]:
for subj in subject_dirs:
  # 24
  if int(subj) == 24:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # # Compute correlation matrix
      # correlation_threshold = 0.5
      # # Extract data as a NumPy array before computing the correlation
      # data = raw.get_data(picks='eeg')
      # ch_names = raw.info['ch_names']
      # corr_matrix = np.corrcoef(data)

      # # Calculate mean correlation for each channel (excluding self-correlation)
      # mean_corrs = []
      # for i in range(corr_matrix.shape[0]):
      #   mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
      #   mean_corrs.append(mean_corr)

      # # Identify bad channels
      # bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      # bad_channels = [ch_names[i] for i in bad_channel_indices]

      # # Output results
      # print(f"Correlation threshold: {correlation_threshold}")
      # print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      # print(bad_channels)

      # # Plot correlation matrix
      # plt.figure(figsize=(16, 14))  # Increased figure size
      # plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      # plt.colorbar(label='Correlation Coefficient')
      # plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # # save
      # # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # # plt.savefig(save_path)
      # # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, duration=60*240)

        # 65


In [ ]:
for subj in subject_dirs:
  # 21
  if int(subj) == 21:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # Compute correlation matrix
      correlation_threshold = 0.6
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, start=15*60, duration=60*60)


In [ ]:
for subj in subject_dirs:
  # 22
  if int(subj) == 22:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # Compute correlation matrix
      correlation_threshold = 0.6
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, start=15*60, duration=60*60)


In [ ]:
for subj in subject_dirs:
  # 25
  if int(subj) == 25:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # Compute correlation matrix
      correlation_threshold = 0.5
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, start=2000, duration=2015)


In [ ]:
for subj in subject_dirs:
  # 26
  if int(subj) == 26:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # Compute correlation matrix
      correlation_threshold = 0.5
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, start=3000, duration=1000)


In [ ]:
for subj in subject_dirs:
  # 27
  if int(subj) == 27:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_001.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      raw.info["bads"] = []

      # Compute correlation matrix
      correlation_threshold = 0.5
      # Extract data as a NumPy array before computing the correlation
      data = raw.get_data(picks='eeg')
      ch_names = raw.info['ch_names']
      corr_matrix = np.corrcoef(data)

      # Calculate mean correlation for each channel (excluding self-correlation)
      mean_corrs = []
      for i in range(corr_matrix.shape[0]):
        mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
        mean_corrs.append(mean_corr)

      # Identify bad channels
      bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
      bad_channels = [ch_names[i] for i in bad_channel_indices]

      # Output results
      print(f"Correlation threshold: {correlation_threshold}")
      print(f"Detected bad channels (mean corr < {correlation_threshold}):")
      print(bad_channels)

      # Plot correlation matrix
      plt.figure(figsize=(16, 14))  # Increased figure size
      plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
      plt.colorbar(label='Correlation Coefficient')
      plt.title(f"Correlation Matrix - Subject {subj} {session_file}\nBad channels: {', '.join(bad_channels) if bad_channels else 'None'}")

      # # save
      # save_path = os.path.join(output_dir, f"{session_file}_corr_matrix.png")
      # plt.savefig(save_path)
      # plt.close()

      for region_name, region_channels in regions.items():
        picks = [ch for ch in region_channels if ch in raw.ch_names]
        # Filter out missing channels
        if not picks:
          print(f"No valid channels found for {region_name} in this file.")
          continue

        # Add enough dummy channels to reach 20
        dummy_index = 1
        while len(picks) < 20:
          dummy_name = f'dummy{dummy_index}'
          if dummy_name not in raw.ch_names:
            dummy_data = np.zeros((1, raw.n_times))
            dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
            dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
            raw.add_channels([dummy_raw])
          picks.append(dummy_name)
          dummy_index += 1

        # Plot
        fig = raw.plot(picks=picks, n_channels=20, scalings=10e-4,
                      title=f"{session_file} - {region_name}", group_by='original',
                      show_scrollbars=False,
                      show_scalebars=False, start=15*60, duration=60*60)


In [ ]:
# Re-reference and filter all subjects
for subj in subject_dirs:
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('.set')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)

      # Load
      raw = mne.io.read_raw_eeglab(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      # Re-referencing
      try:
        raw.set_eeg_reference(ref_channels=['E57', 'E100'])
        print(f"{subj}: Re-referenced using LM/RM average.")
      except Exception as e:
        print(f"{subj}: Error during referencing — {e}")

      # Filtering
      try:
        raw.filter(0.1, None) # high-pass at 0.1 Hz
      except Exception as e:
        print(f"{subj}: Error during filtering - {e}")

      # Save files
      raw.save(os.path.join(output_dir, f"{session_file}_reref_highpass.fif"), overwrite=True)

In [ ]:
# Apply ICA
for subj in subject_dirs:
  # 20
  if int(subj) == 20:
    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_preprocessed.fif')])

    for session_file in session_files:
      session_path = os.path.join(subj_path, session_file)
      # Load
      raw = mne.io.read_raw_fif(session_path, preload=True)
      print(f"Processing: {session_path}")

      # Set standard montage
      raw.set_montage('GSN-HydroCel-128')

      try:
        ica = mne.preprocessing.ICA(random_state=97, method='fastica')
        ica.fit(raw)
      except Exception as e:
        print(f"{subj}: Error during ICA fitting - {e}")
        continue

# save the result before moving forward
# raw.notch_filter(60)  # Notch filter after ICA

      # ICA plots
      try:
        fig_topo = ica.plot_components(picks=range(20), ch_type='eeg', inst=raw, cmap='interactive', colorbar=True, title=f"Participant {subj}: ICA topomap", show=False)
        fig_path_topo = os.path.join(output_dir, f"{subj}_{session_file}_ica_topo.png")
        fig_topo.savefig(fig_path_topo)
        plt.close(fig_topo)

        fig_time = ica.plot_sources(picks=range(20), inst=raw, title=f"Participant {subj}: ICA time series", show=False)
        fig_path_time = os.path.join(output_dir, f"{subj}_{session_file}_ica_time.png")
        fig_time.savefig(fig_path_time)
        plt.close(fig_time)

        print(f"{subj}: Saved ICA plots to {fig_path_topo} and {fig_path_time}")

      except Exception as e:
        print(f"{subj}: Error during ICA plotting - {e}")

In [ ]:
# Manually specify components to exclude based on inspection
# ica.plot_properties(raw, picks=[0, 1, 2, 3, 4])  # Combine view of components 0–4
blink_components = [5]  # replace with indices of artifact components to remove
ica.exclude = blink_components

# Copy original data before applying ICA
raw_cleaned = raw.copy()
ica.apply(raw_cleaned)  # applies ICA, removing selected components

# Save the ICA-cleaned raw object
raw_cleaned.save(os.path.join(output_dir, f"{subj}_{session_file}_ica_cleaned_raw.fif"), overwrite=True)

# Plot original vs cleaned over a short time span
fig_overlay = ica.plot_overlay(raw, exclude=blink_components, picks="eeg", show=False)
overlay_path = os.path.join(output_dir, f"{subj}_{session_file}_overlay.png")
fig_overlay.savefig(overlay_path)
plt.close(fig_overlay)
print(f"{subj}: Saved ICA overlay plot to {overlay_path}")